# Building Your First Network

> 📘 **Python Mastery** · Module 14 — Deep Learning · Lesson 3/7

Everything from Lessons 1–2 snaps together here: you will hand-build, hand-train, and *hand-differentiate* a real classifier on a non-linear problem — then see the same network written in 30 lines of PyTorch.

## 🎯 Learning Objectives

- **Generate** a two-class non-linear dataset (`make_moons`) and inspect why it defeats a straight line.
- **Initialise** network parameters with seeded He initialisation and justify why zeros fail.
- **Write** the forward pass `X@W1+b1 → ReLU → X@W2+b2` as reusable functions.
- **Compute** numerically-stable Binary Cross-Entropy on raw logits.
- **Derive** every backward-pass gradient line-by-line, annotating each with its calculus origin.
- **Train** the network for 300 epochs, report held-out accuracy, and **visualise** its learned decision boundary.

## 1. The Task: Separate Two Moons

Our dataset is two interleaving half-moons — impossible for any straight line (Lesson 1 showed why single neurons can't do this), perfect for one hidden layer. Each point is a "patient" with two measurements; the class is which moon it belongs to.

**Syntax:**
```python
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=200, noise=0.15, random_state=42)   # X:(200,2), y:(200,)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.15, random_state=42)

print("X shape:", X.shape, "| y shape:", y.shape, "| classes:", np.unique(y))
print("first 3 rows:\n", X[:3].round(3))

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X[y == 0, 0], X[y == 0, 1], c="tab:blue", label="class 0", edgecolor="k", s=35)
ax.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:orange", label="class 1", edgecolor="k", s=35)
ax.set_title("Two moons: no straight line can separate these")
ax.set_xlabel("feature 1"); ax.set_ylabel("feature 2")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 2. Step 0 — Split and Standardise the Data

Two habits to build now, forever: hold out a **test set** that the model never trains on, and **standardise features using statistics from the training set only** (applying them to the test set). Leakage of test statistics is a classic subtle bug.

**Syntax:**
```python
idx      = rng.permutation(len(X))     # shuffled row order
X_tr     = (X[idx[:150]] - mu) / sd    # standardise by TRAIN stats
X_te     = (X[idx[150:]] - mu) / sd    # ...applied unchanged to TEST
```

In [ ]:
import numpy as np
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.15, random_state=42)

rng = np.random.default_rng(42)          # seeded shuffle -> reproducible split
idx = rng.permutation(len(X))
X, y = X[idx], y[idx]

n_train = 150
X_train, y_train = X[:n_train], y[:n_train].reshape(-1, 1).astype(float)
X_test,  y_test  = X[n_train:], y[n_train:].reshape(-1, 1).astype(float)

mu, sd = X_train.mean(axis=0), X_train.std(axis=0)   # train statistics ONLY
X_train = (X_train - mu) / sd                        # broadcast standardisation
X_test  = (X_test  - mu) / sd                        # same transform on test

print("train:", X_train.shape, "| test:", X_test.shape)
print("train means after scaling:", X_train.mean(axis=0).round(6))
print("train stds  after scaling:", X_train.std(axis=0).round(3))
print("y_train sample:", y_train.ravel()[:10])

## 3. Step 1 — Initialise Parameters (Seeded)

Weights start small and **random**; biases start at zero. Randomness breaks symmetry (all-hidden-neurons-identical trap from Lesson 1), while the scale $\sqrt{2/\text{fan\_in}}$ (**He initialisation**) keeps signal magnitudes stable layer after layer for ReLU networks.

| Parameter | Shape | Why |
|---|---|---|
| `W1` | `(2, 16)` | One column per hidden neuron, one row per input feature |
| `b1` | `(1, 16)` | Broadcasts down every sample row |
| `W2` | `(16, 1)` | Maps 16 hidden activations to 1 logit |
| `b2` | `(1, 1)` | Output bias |

**Syntax:**
```python
W1 = rng.normal(0, np.sqrt(2 / n_inputs), size=(n_inputs, n_hidden))
```

In [ ]:
import numpy as np

rng = np.random.default_rng(7)                 # separate seed for weights

HIDDEN = 16
W1 = rng.normal(0, np.sqrt(2 / 2),  size=(2, HIDDEN))   # He: fan_in = 2 inputs
b1 = np.zeros((1, HIDDEN))
W2 = rng.normal(0, np.sqrt(2 / HIDDEN), size=(HIDDEN, 1))  # fan_in = 16
b2 = np.zeros((1, 1))

for name, p in [("W1", W1), ("b1", b1), ("W2", W2), ("b2", b2)]:
    print(f"{name}: shape={p.shape}  mean={p.mean():+.4f}  std={p.std():.4f}")

total = sum(p.size for p in [W1, b1, W2, b2])
print(f"total trainable parameters: {total}")

## 4. Step 2 — Forward Pass

The Lesson-1 pipeline, wrapped in functions so training can call it thousands of times. We output raw scores (**logits**) and let the loss handle the sigmoid — the numerically-savvy choice.

$$Z_1 = X W_1 + b_1, \qquad A_1 = \mathrm{ReLU}(Z_1), \qquad Z_2 = A_1 W_2 + b_2$$

**Syntax:**
```python
Z1 = X @ W1 + b1            # (N,2)@(2,H) -> (N,H)
A1 = np.maximum(Z1, 0)      # ReLU
Z2 = A1 @ W2 + b2           # (N,H)@(H,1) -> (N,1)  logits
P  = sigmoid(Z2)            # probabilities, only when needed
```

In [ ]:
import numpy as np

# --- parameters (same init as above; re-run top-to-bottom is fine) ---
rng = np.random.default_rng(7)
HIDDEN = 16
W1 = rng.normal(0, np.sqrt(2 / 2), size=(2, HIDDEN)); b1 = np.zeros((1, HIDDEN))
W2 = rng.normal(0, np.sqrt(2 / HIDDEN), size=(HIDDEN, 1)); b2 = np.zeros((1, 1))

def forward(X):
    """Return intermediates we will need again during backprop."""
    Z1 = X @ W1 + b1                # hidden pre-activations
    A1 = np.maximum(Z1, 0)          # ReLU
    Z2 = A1 @ W2 + b2               # output logits
    return Z1, A1, Z2

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

demo = np.array([[0.0, 1.0], [1.5, -0.5]])
Z1, A1, Z2 = forward(demo)
print("Z1 shape", Z1.shape, "-> A1 kept", (A1 > 0).sum(), "of",
      A1.size, "units alive (ReLU)")
print("logits:", Z2.ravel().round(4))
print("probs :", sigmoid(Z2).ravel().round(4))

## 5. Step 3 — Loss: Binary Cross-Entropy on Logits

We could apply sigmoid then plug into Lesson 1's BCE — but exponentials overflow and logs explode near 0/1. The production-grade form folds the sigmoid *into* the loss algebraically and stays stable for any logit:

$$L = \frac{1}{N}\sum_{i} \Big[ \max(z_i, 0) \;\;-\;\; y_i z_i \;\;+\;\; \log\big(1 + e^{-|z_i|}\big) \Big]$$

Same value as $-\big[y\log p + (1-y)\log(1-p)\big]$ with $p=\sigma(z)$, but no $\infty$s. This exact formula ships as `nn.BCEWithLogitsLoss`.

**Syntax:**
```python
loss = np.mean(np.clip(np.logaddexp(0, z), None, 30) - y * z)
# logaddexp(0, z) = log(1 + e^z): softplus, always finite
```

In [ ]:
import numpy as np

def bce_with_logits(z, y):
    """Numerically-stable BCE for raw logits z and labels y in {0, 1}."""
    return np.mean(
        np.maximum(z, 0)              # term 1: max(z, 0)
        - y * z                       # term 2: -y*z  (pulls correct-class logit up)
        + np.log1p(np.exp(-np.abs(z)))  # term 3: log(1 + e^-|z|), never overflows
    )

y_true = np.array([[1.0], [0.0]])

good = np.array([[+3.0], [-3.0]])    # confident AND correct
bad  = np.array([[-3.0], [+3.0]])    # confident AND wrong
zero = np.array([[0.0], [0.0]])      # completely undecided

for name, z in [("confident & correct", good),
                ("undecided (z=0)", zero),
                ("confident & WRONG ", bad)]:
    p = 1 / (1 + np.exp(-z))
    print(f"{name}: loss={bce_with_logits(z, y_true):.4f}  (p={p.ravel().round(3)})")

print("\n-> the further the logit sits from the truth, the bigger the penalty.")

## 6. Step 4 — Backpropagation, Layer by Layer

Now the payoff of Lesson 1's chain rule. Blame flows backwards: loss → logits → hidden activations → first-layer weights. Every line below states *which* derivative rule produced it.

| Gradient | Formula | Calculus origin |
|---|---|---|
| Output error | $dZ_2 = (\sigma(Z_2) - y)/N$ | $\partial L/\partial z$ of BCE-with-logits — sigmoid' cancels leaving $\sigma(z)-y$; $/N$ from the mean |
| `dW2` | $A_1^\top\, dZ_2$ | $Z_2 = A_1W_2+b_2 \Rightarrow \partial Z_2/\partial W_2 = A_1$ |
| `db2` | $\sum dZ_2$ | bias enters additively per sample |
| Hidden error | $dA_1 = dZ_2\, W_2^\top$ | route blame back across $W_2$: transpose flips the mapping |
| `dZ1` | $dA_1 \odot \mathbb{1}[Z_1>0]$ | ReLU': gradient passes where positive, else 0 |
| `dW1` | $X^\top\, dZ_1$, `db1` $=\sum dZ_1$ | same pattern as layer 2 |

Notice layers 1 and 2 share one template: **gradient wrt W = (input activations)ᵀ @ (output error)**. That template repeats in every network you will ever build.

**Syntax:**
```python
dZ2 = (sigmoid(Z2) - y) / N       # 1. output error
dW2 = A1.T @ dZ2                  # 2. second-layer weights
db2 = dZ2.sum(0, keepdims=True)   # 3. second-layer bias
dA1 = dZ2 @ W2.T                  # 4. send blame to hidden units
dZ1 = dA1 * (Z1 > 0)              # 5. through ReLU's gate
dW1 = X.T @ dZ1                   # 6. first-layer weights
db1 = dZ1.sum(0, keepdims=True)   # 7. first-layer bias
```

In [ ]:
import numpy as np

# --- rebuild the tiny setup so this cell runs standalone ---
rng = np.random.default_rng(7)
W1 = rng.normal(0, np.sqrt(2 / 2), size=(2, 8)); b1 = np.zeros((1, 8))
W2 = rng.normal(0, np.sqrt(2 / 8), size=(8, 1)); b2 = np.zeros((1, 1))

Xs = np.array([[0.5, -1.0], [1.5, 0.5]])          # 2 demo samples
ys = np.array([[1.0], [0.0]])
N = len(Xs)

def sigmoid(z): return 1 / (1 + np.exp(-z))

# FORWARD (keep everything!)
Z1 = Xs @ W1 + b1
A1 = np.maximum(Z1, 0)
Z2 = A1 @ W2 + b2

# BACKWARD -- every line annotated with its calculus origin
sig_z2 = sigmoid(Z2)
dZ2 = (sig_z2 - ys) / N            # dL/dZ2 : BCE-on-logits collapses to (sigma - y)/N
dW2 = A1.T @ dZ2                   # dL/dW2 : z2 = A1 @ W2  => grad = A1^T @ upstream
db2 = dZ2.sum(0, keepdims=True)    # dL/db2 : additive bias => sum of upstream
dA1 = dZ2 @ W2.T                   # dL/dA1 : transpose routes blame back through W2
dZ1 = dA1 * (Z1 > 0)               # dL/dZ1 : ReLU' = indicator(Z1 > 0), elementwise gate
dW1 = Xs.T @ dZ1                   # dL/dW1 : z1 = X @ W1   => grad = X^T @ upstream
db1 = dZ1.sum(0, keepdims=True)    # dL/db1 : sum of upstream, same as db2

for name, g in [("dW2", dW2.ravel()[:4]), ("dZ2", dZ2.ravel()),
                ("dW1[0,:4]", dW1[0, :4]), ("db1", db1.ravel()[:4])]:
    print(f"{name:10s}", g.round(5))
print("shapes:", dW1.shape, db1.shape, dW2.shape, db2.shape,
      "<- every gradient mirrors its parameter")

> 🔍 **Under the Hood:** when PyTorch runs your forward pass it stores exactly these intermediates (`Z1`, `A1`, ...) plus each op's backward recipe. `loss.backward()` replays the table above in reverse order — the `(σ(z)-y)` trick, the transposes, the ReLU mask — for networks with billions of parameters. You are about to run it manually once; afterwards, framework magic should feel like familiar bookkeeping.

## 7. Step 5 — The Training Loop

Assemble the machine: forward → loss → backward → update ($w \leftarrow w - \text{lr}\cdot g$), repeated. With **full-batch** gradient descent, one update consumes all 150 training rows per epoch — slow per step, beautifully smooth curve.

**Syntax:**
```python
for epoch in range(300):
    Z1, A1, Z2 = forward(X_train)        # 1 predict
    loss   = bce_with_logits(Z2, y_train)  # 2 measure
    ...                                    # 3 compute gradients
    W1 -= lr * dW1                          # 4 descend
```

In [ ]:
import numpy as np
from sklearn.datasets import make_moons

# ---------- data ----------
X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
rng = np.random.default_rng(42)
idx = rng.permutation(len(X))
X, y = X[idx], y[idx]
X_train, y_train = X[:150], y[:150].reshape(-1, 1).astype(float)
X_test,  y_test  = X[150:], y[150:].reshape(-1, 1).astype(float)
mu, sd = X_train.mean(0), X_train.std(0)
X_train, X_test = (X_train - mu) / sd, (X_test - mu) / sd

# ---------- helpers ----------
def sigmoid(z): return 1 / (1 + np.exp(-z))
def bce_logits(z, y):
    return np.mean(np.maximum(z, 0) - y * z + np.log1p(np.exp(-np.abs(z))))

# ---------- parameters (seeded He init) ----------
r = np.random.default_rng(7)
HID = 16
W1 = r.normal(0, np.sqrt(2 / 2), (2, HID)); b1 = np.zeros((1, HID))
W2 = r.normal(0, np.sqrt(2 / HID), (HID, 1)); b2 = np.zeros((1, 1))

lr = 0.5                                        # stride length (Lesson 1)
for epoch in range(1, 301):
    # ---- forward ----
    Z1 = X_train @ W1 + b1
    A1 = np.maximum(Z1, 0)
    Z2 = A1 @ W2 + b2
    loss = bce_logits(Z2, y_train)

    # ---- backward (Section 6 template) ----
    dZ2 = (sigmoid(Z2) - y_train) / len(y_train)
    dW2 = A1.T @ dZ2;                    db2 = dZ2.sum(0, keepdims=True)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (Z1 > 0)
    dW1 = X_train.T @ dZ1;               db1 = dZ1.sum(0, keepdims=True)

    # ---- update ----
    W1 -= lr * dW1; b1 -= lr * db1; W2 -= lr * dW2; b2 -= lr * db2

    if epoch % 50 == 0:
        print(f"epoch {epoch:3d} | train loss {loss:.4f}")

# ---- evaluate ONCE on the untouched test set ----
Z1t = np.maximum(X_test @ W1 + b1, 0)
Z2t = Z1t @ W2 + b2
acc = ((Z2t.ravel() > 0) == y_test.ravel()).mean()
print(f"\nFINAL test accuracy: {acc:.1%}  <- a straight line scored ~85% at best")

## 8. The Finale — Watching the Decision Boundary It Learned

The network is just a function $f(x_1, x_2) \mapsto [0,1]$. Evaluate it on a dense grid covering the plane, colour each grid cell by its prediction, and overlay the data: the curved boundary that defeated our perceptron appears. *This picture is what "learning a representation" looks like.*

**Syntax:**
```python
grid = np.c_[gx.ravel(), gy.ravel()]     # meshgrid -> (G*G, 2) points
probs = sigmoid(forward(grid)[2]).reshape(gx.shape)
plt.contourf(gx, gy, probs, levels=20, cmap="RdBu")   # colour the plane
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# ---- retrain quickly so this cell stands alone (same seeds -> same net) ----
X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
rs = np.random.default_rng(42); idx = rs.permutation(len(X))
X, y = X[idx], y[idx]
Xtr, ytr = X[:150], y[:150].reshape(-1, 1).astype(float)
Xte, yte = X[150:], y[150:].reshape(-1, 1).astype(float)
mu, sd = Xtr.mean(0), Xtr.std(0)
Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd

def sigmoid(z): return 1 / (1 + np.exp(-z))
r = np.random.default_rng(7); HID = 16
W1 = r.normal(0, 1.0, (2, HID)); b1 = np.zeros((1, HID))
W2 = r.normal(0, np.sqrt(1 / HID), (HID, 1)); b2 = np.zeros((1, 1))
for _ in range(300):
    Z1 = Xtr @ W1 + b1; A1 = np.maximum(Z1, 0); Z2 = A1 @ W2 + b2
    dZ2 = (sigmoid(Z2) - ytr) / len(ytr)
    dW2 = A1.T @ dZ2; db2 = dZ2.sum(0, keepdims=True)
    dZ1 = (dZ2 @ W2.T) * (Z1 > 0)
    dW1 = Xtr.T @ dZ1; db1 = dZ1.sum(0, keepdims=True)
    W1 -= 0.5 * dW1; b1 -= 0.5 * db1; W2 -= 0.5 * dW2; b2 -= 0.5 * db2

# ---- paint the plane ----
gx, gy = np.meshgrid(np.linspace(-2.5, 2.5, 300), np.linspace(-2.5, 2.5, 300))
grid = np.c_[gx.ravel(), gy.ravel()]
hidden = np.maximum(grid @ W1 + b1, 0)
probs = sigmoid(hidden @ W2 + b2).reshape(gx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
cf = ax.contourf(gx, gy, probs, levels=25, cmap="RdBu", vmin=0, vmax=1)
ax.contour(gx, gy, probs, levels=[0.5], colors="black", linewidths=1.5)
fig.colorbar(cf, ax=ax, label="P(class 1)")

ax.scatter(Xtr[ytr.ravel() == 0][:, 0], Xtr[ytr.ravel() == 0][:, 1],
           c="tab:blue", edgecolor="k", s=30, label="train class 0")
ax.scatter(Xtr[ytr.ravel() == 1][:, 0], Xtr[ytr.ravel() == 1][:, 1],
           c="tab:orange", edgecolor="k", s=30, label="train class 1")
ax.scatter(Xte[:, 0], Xte[:, 1], c=yte.ravel(), cmap="coolwarm",
           marker="*", s=90, edgecolor="k", label="test points")

ax.set_title("Decision boundary learned by our 2-16-1 NumPy network")
ax.set_xlabel("feature 1 (standardised)"); ax.set_ylabel("feature 2 (standardised)")
ax.legend(loc="upper left", fontsize=9)
plt.show()

print("Black line = P(class 1) = 0.5. Notice how it bends around BOTH moons.")

## 9. Training Vocabulary You Will Hear Everywhere

| Term | Meaning | In the loop above |
|---|---|---|
| **Epoch** | One complete pass over the training data | Each of our 300 loops = 1 epoch (we saw all 150 rows) |
| **Batch / mini-batch** | Subset of data used for one update | We used ALL rows: *full-batch* gradient descent |
| **Step / iteration** | One parameter update | 300 steps total (1 per epoch here) |
| **Batch size** | Samples per step | 150 (full-batch); typically 32–512 |
| **Stochastic GD (SGD)** | Updates per *mini-batch*, not whole set | Next lesson's default mode |

With batch size 32 and 150 samples, one epoch = ⌈150/32⌉ = 5 steps — epochs ≠ steps once batching begins.

## 10. The Same Network in PyTorch

Every manual step above has a one-line framework counterpart. Compare carefully — the vocabulary maps 1:1.

**PyTorch version** (requires `pip install torch`)
```python
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1) data -> tensors -> mini-batches  (replaces full-batch numpy slicing)
X_t = torch.tensor(X_train, dtype=torch.float32)
y_t = torch.tensor(y_train, dtype=torch.float32)
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=32, shuffle=True)

# 2) model: Linear(2->16) IS our W1/b1, Linear(16->1) IS our W2/b2
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1)).to(device)

opt = torch.optim.Adam(model.parameters(), lr=0.01)   # smarter descent (next lesson)
loss_fn = nn.BCEWithLogitsLoss()                      # our stable BCE, built in

for epoch in range(300):                              # 3) training loop
    for xb, yb in loader:
        opt.zero_grad()                               # clear .grad (it accumulates!)
        loss = loss_fn(model(xb), yb)                 # forward pass
        loss.backward()                               # <-- Sections 6 happens HERE, automatically
        opt.step()                                    # parameter update (w -= lr * grad)
    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch+1:3d} | last batch loss {loss.item():.4f}")

with torch.no_grad():                                 # inference: skip graph recording
    logits = model(torch.tensor(X_test, dtype=torch.float32)).squeeze()
    preds = (logits.cpu().numpy() > 0).astype(int)
    print("test accuracy:", round((preds == y_test.ravel()).mean(), 3))
print("test accuracy:", round(acc, 3))
```

Roughly 30 lines, identical maths — and `loss.backward()` deleted our entire Section 6.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Dropping the `/ N` in `dZ2` | Effective learning rate scales with dataset size — code that worked on 200 rows explodes on 20 000 | Divide summed-loss gradients by batch size (or use `.mean()` semantics consistently) |
| `X.T @ dZ` orientation errors | Cryptic "mat1 and mat2 shapes cannot be multiplied" | Rule: **(inputs to the layer)ᵀ @ (error flowing out)**; verify inner dims match before multiplying |
| Applying sigmoid *and* using `BCEWithLogitsLoss` | Double squashing saturates gradients; loss barely moves | Feed **raw logits** to `BCEWithLogitsLoss`; sigmoid only for reporting probabilities |
| Standardising test data with test-set mean/std | Leaks information; inflated scores | Compute `mu, sd` on train, apply to both splits |
| Judging success on training accuracy alone | Memorisation masquerades as learning | Report held-out test metrics; watch the train/test gap (next lesson) |
| Unseeded init/shuffle | "Works on my machine" bugs nobody can reproduce | Seed `np.random.default_rng(...)` / `torch.manual_seed(...)` |

## 💡 Best Practices & Pro Tips

- **Build forward and backward side by side**, checking that every gradient's shape equals its parameter's shape — mismatched shapes are caught instantly, mismatched maths silently.
- **Keep the loss printed every few epochs**: a healthy curve falls fast then flattens; spikes mean lower the lr, flatness means raise capacity or check data.
- **Visualise what your model learned** whenever dimensions allow — a decision boundary (or attention map, or saliency image) teaches more than any metric.
- **Start full-batch, then go mini-batch:** full-batch removes shuffle-noise while you debug maths; switch to mini-batches for speed and better generalisation at scale.
- **AI-engineering relevance:** this exact skeleton — dataset, model, loss, optimiser, loop, eval — is the universal training script you will read in every production repo; only the model block changes between a moon classifier and GPT.

## 📌 Summary

| Piece | Our implementation | Framework equivalent |
|---|---|---|
| Dataset | `make_moons`, standardised | `Dataset` / `DataLoader` |
| Model | `X@W1+b1 → ReLU → @W2+b2` | `nn.Sequential(nn.Linear, nn.ReLU, nn.Linear)` |
| Init | `rng.normal(0, sqrt(2/fan_in))` | default Kaiming init in `nn.Linear` |
| Loss | `max(z,0) − y·z + log1p(exp(−|z|))` | `nn.BCEWithLogitsLoss()` |
| Backward | 7 hand-derived lines | `loss.backward()` |
| Update | `W -= lr * dW` | `optimizer.step()` |

Key takeaways:
- A working neural net needs only: data → forward → loss → gradients → update, looping.
- Backprop follows one repeating pattern — layer inputᵀ @ outgoing error — regardless of depth.
- Held-out evaluation and visualisation turn "loss went down" into "the model actually learned the right thing".
- Frameworks automate the mechanics, not the thinking: knowing the 7 gradient lines is what lets you debug when they misbehave.

## 🔗 Next Lesson

**04_Training_Optimization** — mini-batches, momentum-based optimisers, learning-rate schedules, and regularisation: making training faster, stabler, and honest.